# 01 — Explore the House M.D. OpenEnv

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sneh2909/Overfitters/blob/main/notebooks/01_explore_env.ipynb)
[![Open in Spaces](https://huggingface.co/datasets/huggingface/badges/resolve/main/open-in-hf-spaces-sm.svg)](https://huggingface.co/spaces/SnehShah/house-md-env)

This notebook is the **judge entry point**. **No GPU required** — it runs on a free Colab CPU runtime in ~3 minutes.

It connects to the deployed environment (`SnehShah/house-md-env` on Hugging Face Spaces) via the OpenEnv `HouseMDEnv` client and walks through:

1. The `Action` / `Observation` / `State` schemas
2. A short manual episode against the live Space
3. A random-policy episode (baseline)
4. An oracle-policy episode (upper bound)
5. Reward breakdown across the five active rubrics

> **Live UI**: The same Space also serves the animated ER triage scene at the root URL — open <https://snehshah-house-md-env.hf.space/> in a new tab if you want to see the same patient through a clinician's eyes.

## 1. Install the OpenEnv client + utilities

We pull the **client side only** of the env package straight from the GitHub repo. No CUDA, no model weights, no patient data — the live Space holds all of that.

In [ ]:
%%capture
!pip install -q "openenv-core>=0.2.2" requests pydantic
!pip install -q "git+https://github.com/sneh2909/Overfitters.git#subdirectory=house_md_env"

## 2. Connect to the live Space

`HouseMDEnv` is a thin OpenEnv client. `.sync()` opens a session against the FastAPI server running inside the Space.

In [ ]:
import requests
from house_md_env import HouseMDEnv, HouseMDAction, HouseMDActionType, HouseMDObservation, HouseMDState

ENV_URL = "https://snehshah-house-md-env.hf.space"

env = HouseMDEnv(base_url=ENV_URL).sync()
print("connected to:", ENV_URL)
print("metadata    :", requests.get(ENV_URL + "/metadata").json())

## 3. Inspect the action / observation schema

The action space has 5 verbs (closed vocabulary; the `argument` enum is fixed):

| ActionType | argument enum | typical cost | latency |
| --- | --- | --- | --- |
| `INTERVIEW` | 25 question ids | $0 | 1 step |
| `EXAMINE` | 15 exam ids | low | 1 step |
| `ORDER_TEST` | 35 lab/imaging ids | $5 – $1500 | 1–3 steps (delayed) |
| `UPDATE_DIFFERENTIAL` | free-text summary + board[] | $0 | instant |
| `DIAGNOSE` | 35 disease ids | $0 | terminal |

In [ ]:
print("ActionType options:")
for at in HouseMDActionType:
    print(f"  - {at.value}")

import json
print("\nHouseMDAction schema (Pydantic):")
print(json.dumps(HouseMDAction.model_json_schema(), indent=2)[:600] + "\n  ...")

## 4. Reset to a new patient and read the intake

Disease / variant / seed are sampled by the server. The intake gives us **noisy partial information** — the agent has to dig for the rest.

In [ ]:
result = env.reset()
obs: HouseMDObservation = result.observation

print("=== INTAKE ===")
print(f"patient          : {obs.age}y {obs.sex}")
print(f"chief complaint  : {obs.chief_complaint}")
print(f"intake vitals    : {obs.intake_vitals}")
print()
print(f"step / step_cap  : {obs.step} / {obs.step_cap}")
print(f"cost so far ($)  : {obs.cost_so_far}")
print(f"time elapsed (m) : {obs.time_elapsed_min}")
print(f"severity signal  : {obs.severity_signal}")
print(f"differential     : {obs.differential_board}")
print(f"terminal         : {obs.terminal}")

## 5. Step manually — ask one question

We send `INTERVIEW(pain_onset)` and inspect the noisy patient response.

In [ ]:
action = HouseMDAction(
    type=HouseMDActionType.INTERVIEW,
    argument="pain_onset",
    rationale="Time course narrows acute vs chronic differentials",
)
step_result = env.step(action)
obs = step_result.observation

last = obs.action_log[-1]
print(f"action       : {last.type} / {last.argument}")
print(f"response     : {last.text}")
print(f"step reward  : {step_result.reward}")
print(f"cost so far  : ${obs.cost_so_far}")
print(f"step / cap   : {obs.step} / {obs.step_cap}")

## 6. Run a full **random-policy** episode

This is our floor baseline. The 45-patient eval result on the random policy is in `results/eval_results.json`; here we just show the API contract.

In [ ]:
import random

ARG_MENU = {
    HouseMDActionType.INTERVIEW: [
        "pain_onset", "pain_location", "pain_character",
        "past_medical_history", "family_history",
    ],
    HouseMDActionType.EXAMINE: [
        "vital_signs", "abdominal_palpation", "cardiac_auscultation",
    ],
    HouseMDActionType.ORDER_TEST: [
        "cbc", "bmp", "ecg", "chest_xray",
    ],
    HouseMDActionType.DIAGNOSE: [
        "appendicitis", "stemi", "migraine", "viral_uri", "pneumonia",
    ],
}

obs = env.reset().observation
for t in range(8):
    at = random.choice([HouseMDActionType.INTERVIEW, HouseMDActionType.EXAMINE, HouseMDActionType.ORDER_TEST])
    arg = random.choice(ARG_MENU[at])
    res = env.step(HouseMDAction(type=at, argument=arg, rationale="random policy"))
    obs = res.observation
    if obs.terminal:
        break

if not obs.terminal:
    diag = HouseMDAction(
        type=HouseMDActionType.DIAGNOSE,
        argument=random.choice(ARG_MENU[HouseMDActionType.DIAGNOSE]),
        rationale="random guess",
    )
    obs = env.step(diag).observation

rewards = obs.rewards or {}
print(f"steps used : {obs.step}")
print(f"cost       : ${obs.cost_so_far}")
print(f"diagnosis  : {obs.diagnosis}")
print(f"timed out  : {obs.timed_out}")
print(f"rewards    : {rewards}")
print(f"total      : {rewards.get('total', 0.0):+.3f}")

## 7. Reward breakdown

The Space exposes five active reward rubrics (full design lives in `clinical_rl/rewards.py`):

| key | range | meaning |
| --- | --- | --- |
| `r1_accuracy` | `[-2, +1]` | terminal: did the diagnosis match the ground truth disease? |
| `r2_cost` | `[-1.5, +1]` | how much money did we burn vs the oracle ceiling? |
| `r6_anchoring` | `[-0.5, +0.6]` | did we update the differential as evidence arrived? |
| `r7_safety` | `[-2, 0]` | did we miss a critical / time-sensitive condition? |
| `r8_format` | `[0, 1]` | were our action JSONs well-formed? |
| `total` | weighted sum | what GRPO actually optimises |

In [ ]:
for k, v in (obs.rewards or {}).items():
    print(f"  {k:<14} {v:>+7.3f}")

## 8. Oracle-policy episode (upper bound)

The Space also exposes a session-based **oracle policy** at `/api/episodes/<sid>/agent_step?policy=oracle` (used by the live ER UI when a clinician clicks *“Auto-step (oracle)”*). It plays the same patient using the heuristic `HeuristicOracle` from the codebase — a sensible upper bound on what an SFT/GRPO agent should approach.

This cell **opens a separate session** through the playground API so we don't disturb our OpenEnv `env` handle above.

In [ ]:
API = ENV_URL + "/api"
session = requests.post(f"{API}/episodes", json={"disease": "appendicitis", "variant_id": "v1", "seed": 42}).json()
sid = session["session_id"]
print(f"new oracle session: {sid}  disease=appendicitis")

obs_json = session
for _ in range(15):
    obs_json = requests.post(f"{API}/episodes/{sid}/agent_step", json={"policy": "oracle"}).json()
    if obs_json.get("terminal"):
        break

rewards = requests.get(f"{API}/episodes/{sid}/rewards").json()
truth = requests.get(f"{API}/episodes/{sid}/truth").json()
print(f"\nground truth disease : {truth.get('true_disease_name')} ({truth.get('true_disease_id')})")
print(f"oracle diagnosis     : {obs_json.get('diagnosis')}")
print(f"oracle steps used    : {obs_json.get('step')}")
print(f"oracle cost          : ${obs_json.get('cost_so_far')}")
print(f"oracle rewards       : {rewards}")

## 9. Where to go next

* **`02_sft.ipynb`** – fine-tune Gemma 3 4B-IT on oracle traces (LoRA + Unsloth + TRL `SFTTrainer`).
* **`03_grpo.ipynb`** – run a GRPO loop against this same live Space and reproduce the W&B curves below.
* **`04_eval_compare.ipynb`** – score base / SFT / GRPO on the held-out 45-patient eval split and produce the comparison plots.
* **W&B run**: <https://wandb.ai/sneh2909-christ-university/house-md?nw=nwusersneh2909>

Tear down the session when you're done so we free a slot on the Space:

In [ ]:
env.close()
requests.delete(f"{API}/episodes/{sid}")
print("sessions closed")